# Evaluation: throughput vs time (per path, paper-style)

- **X**: time (s) from the first `path :* mean tp` line in the pull log  
- **Y**: per-path `mean tp` (Mbps) from the same log — typically **path 0 / path 1** = the two subflows  
- **Shaded bands**: `tc_bw_*.log` — capacity on the **shaped** interface (e.g. h1-eth0); not the same as “sum of two links’ budgets”  
- **Vertical lines**: time of each `tc` step (e.g. 0 s, 50 s, 100 s)  

Set `RUN_DIR` to a folder that contains `pull_*.log` and `tc_bw_*.log`.  
Run the first cells from the **repository root** (or adjust `REPO` below). Later you can add mechanism layers (U, gain, …) in new cells.

In [ ]:
import os
import re
import sys
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd


def find_repo() -> Path:
    """Cwd or parents until scripts/analyze/parse_logs.py exists."""
    start = Path.cwd().resolve()
    for p in [start, *start.parents]:
        if (p / "scripts" / "analyze" / "parse_logs.py").is_file():
            return p
    return start


REPO = find_repo()
sys.path.insert(0, str(REPO / "scripts" / "analyze"))
import parse_logs as pl  # noqa: E402

print("REPO =", REPO)

In [ ]:
# --- run directory (one vm_run folder with pull + tc_bw) ---
RUN_DIR = REPO / "logs_exp" / "log" / "vm_run_20260426_033221"  # <- change to your run

PULL = next(RUN_DIR.glob("pull_*.log"), None)
TC_BW = next(RUN_DIR.glob("tc_bw_*.log"), None)
assert PULL and TC_BW, f"Need pull_*.log and tc_bw_*.log under {RUN_DIR}"
PULL, TC_BW

In [ ]:
_RE_MEAN_TP = re.compile(
    r"^(?P<date>\d{4}/\d{2}/\d{2}) (?P<time>\d{2}:\d{2}:\d{2}) path :(?P<path>\d) mean tp: (?P<tp>[\d.]+)Mbps"
)


def load_mean_tp_by_path(pull_path: Path) -> tuple[pd.DataFrame, float | None]:
    """Return long-form df [t_sec, path, tp_mbps] and t0 (unix of first row)."""
    rows: list[dict] = []
    t0: float | None = None
    with open(pull_path, encoding="utf-8", errors="replace") as f:
        for line in f:
            m = _RE_MEAN_TP.match(line)
            if not m:
                continue
            try:
                tp = float(m["tp"])
            except ValueError:
                continue
            if not (tp == tp) or tp < 0:
                continue
            dt = datetime.strptime(m["date"] + " " + m["time"], "%Y/%m/%d %H:%M:%S")
            ts = dt.timestamp()
            if t0 is None:
                t0 = ts
            t_sec = ts - t0
            rows.append({"t_sec": t_sec, "path": int(m["path"]), "tp_mbps": tp})
    return pd.DataFrame(rows), t0


def wide_per_path(long_df: pd.DataFrame) -> pd.DataFrame:
    """Columns t_sec, p0, p1, ... and total."""
    if long_df.empty:
        return pd.DataFrame()
    w = long_df.pivot_table(index="t_sec", columns="path", values="tp_mbps", aggfunc="mean")
    w = w.sort_index().reset_index()
    w.columns = ["t_sec"] + [f"path_{c}" for c in w.columns[1:]]
    num_cols = [c for c in w.columns if c != "t_sec"]
    w["total"] = w[num_cols].sum(axis=1, min_count=1)
    return w


long_df, t0 = load_mean_tp_by_path(PULL)
wide = wide_per_path(long_df)
long_df.head(), wide.head()

In [ ]:
# Optional smoothing (set to 1 to disable)
SMOOTH = 5

if not wide.empty and SMOOTH > 1:
    wplot = wide.copy()
    for c in wplot.columns:
        if c == "t_sec":
            continue
        wplot[c] = wplot[c].rolling(SMOOTH, min_periods=1).mean()
else:
    wplot = wide

t_max = float(wplot["t_sec"].max()) if not wplot.empty else 0.0

In [ ]:
# --- tc step times (pull-coordinate) and shaped-link regions ---
tc_df = pl.tc_bw_with_pull_t(TC_BW, PULL).sort_values("at_sec").reset_index(drop=True)
tc_df

In [ ]:
from matplotlib.patches import Patch

fig, ax = plt.subplots(figsize=(11, 4.2))
band_colors = ["#fff3cd", "#f8d7da", "#d4edda", "#cce5ff"]
legend_patches: list[tuple[object, str]] = []

if not tc_df.empty and tc_df["t_pull"].notna().any():
    n = len(tc_df)
    for i in range(n):
        t_start = float(tc_df.loc[i, "t_pull"])
        t_end = float(tc_df.loc[i + 1, "t_pull"]) if i + 1 < n else t_max + 1.0
        bw = float(tc_df.loc[i, "bw_mbit"])
        dev = str(tc_df.loc[i, "dev"])
        t_end = min(t_end, t_max + 0.1)
        ax.axvspan(t_start, t_end, color=band_colors[i % len(band_colors)], alpha=0.4, zorder=0)
        at_sec = float(tc_df.loc[i, "at_sec"])
        legend_patches.append(
            (Patch(facecolor=band_colors[i % len(band_colors)], alpha=0.4, edgecolor="none"), f"{bw:.0f} Mbit/s from t≈{at_sec:.0f}s ({dev})")
        )
    t0s = [float(x) for x in tc_df["t_pull"].values if x == x]
    for t_step in t0s:
        ax.axvline(t_step, color="#333", ls="--", lw=0.8, zorder=1, alpha=0.7)
else:
    pass

pathCols = [c for c in wplot.columns if c.startswith("path_")]
styles = ["-", "-", "-", ":"]
for i, col in enumerate(pathCols):
    label = f"{col.replace('_', ' ')} (mean tp)"
    ax.plot(wplot["t_sec"], wplot[col], label=label, ls=styles[i % len(styles)], lw=1.1, zorder=3)
if "total" in wplot.columns:
    ax.plot(wplot["t_sec"], wplot["total"], label="total (path sum)", color="k", ls="-.", lw=1.0, zorder=3, alpha=0.85)

handles, labels = ax.get_legend_handles_labels()
for p, lab in legend_patches:
    handles.append(p)
    labels.append(lab)
ax.legend(handles, labels, loc="upper right", fontsize=8, framealpha=0.9)

ax.set_xlabel("Time (s) from first mean-tp sample in pull log")
ax.set_ylabel("Throughput (Mbps)")
ax.set_title("Per-path mean tp + total; shaded: shaped-link TBF; dashed: step times")
ax.grid(True, alpha=0.3, zorder=2)
ax.set_xlim(0, max(t_max * 1.02, 1.0))
fig.tight_layout()
out_png = REPO / "derived" / "evaluation_timeline_notebook.png"
out_png.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(out_png, dpi=150)
plt.show()
print("Saved:", out_png)

## Notes

- **Legend (lines)**: one curve per `path` in the log + optional **total**.
- **Legend (patches)**: each coloured band = **TBF cap on the shaped interface** for that time interval; compare with the paper’s “link1/link2” **only** if you map `path0/path1` to h1-eth0/eth1 the same way in your run.
- If **one path** stays ~0, one line will hug the x-axis: that is expected when almost all data uses the other path.

Next: copy new cells for mechanism plots (`[utility]`, `[m]monitor`, `verify_projected_gradient`, etc.).